In [ ]:
import torch
from torch import Tensor
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

def byte_pair_encoding(text, vocabulary_size):
    vocabulary = {}
    most_frequent = None
    replacement = 888 # First unused unicode codepoint
    tokens = text.encode("utf-8") # Each token is a scalar

    # Tokenize text into a list of iteratively compressed bytes
    while len(vocabulary) < vocabulary_size:
        for i in range(0, len(tokens) - 1):
            group = (tokens[i], tokens[i + 1])
            if group not in vocabulary:
                vocabulary[group] = [i]
            else:
                vocabulary[group].append(i)

            if most_frequent is None or len(vocabulary[group]) > len(vocabulary[most_frequent]):
                most_frequent = group

        indexes = vocabulary[most_frequent]
        for i in indexes:
            tokens = [*tokens[:i], replacement, *tokens[i + 2:]]
        replacement += 1

    return tokens, vocabulary

class WMT(Dataset):
    def __init__(self, data_split, num_rows, vocabulary_size):
        super().__init__()
        dataset = load_dataset("wmt/wmt14", "fr-en", split=f"{data_split}[:{num_rows}]")
        self.df = dataset.data.to_pandas()
        self.rows: list[tuple[Tensor, Tensor, dict, dict] | None] = [None for _ in range(num_rows)]
        self.vocab_size = vocabulary_size

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, idx):
        if self.rows[idx] is None:
            c = self.df.columns[0]
            en, fr = self.df[c].str["en"][idx], self.df[c].str["fr"][idx]
            en_tokens, en_vocabulary = byte_pair_encoding(en, self.vocab_size)
            fr_tokens, fr_vocabulary = byte_pair_encoding(fr, self.vocab_size)
            self.rows[idx] = (
                torch.tensor(en_tokens), torch.tensor(fr_tokens),
                en_vocabulary, fr_vocabulary)
        return self.rows[idx][0], self.rows[idx][1]

/home/aabiji/dev/ml/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
train_samples = WMT("train", 64, 10)
# TODO: Add start token and end token, then pad each token up to a set size, although it will take more memory
#       First, find out what the max token size is, then introduce another mask on the similarity matrix to mask out the padding for each token.
train_loader = DataLoader(dataset=train_samples, batch_size=1, shuffle=True)
for en_token, fr_token in train_loader:
    print(en_token, fr_token)
"""

In [101]:
class MultiHeadAttention(nn.Module):
    def __init__(self, B, d_model, d_k, d_v, h):
        super().__init__()
        self.W_q = torch.rand(B, h, d_model, d_k)
        self.W_k = torch.rand(B, h, d_model, d_k)
        self.W_v = torch.rand(B, h, d_model, d_v)
        self.W_o = torch.rand(B, h * d_v, d_model)

    def forward(self, Q, K, V):
        # Project Q, K, V into smaller subspaces using each h projection matrices
        Q_proj = torch.einsum("bij,bhjk->bhik", Q, self.W_q)
        K_proj = torch.einsum("bij,bhjk->bhik", K, self.W_k)
        V_proj = torch.einsum("bij,bhjk->bhik", V, self.W_v)

        # Compute attention for each attention head:
        # Q @ K.T for each attention head in each batch to get similarity matrices
        a = torch.einsum("bhij,bhkj->bhik", Q_proj, K_proj)
        # Scale by 1 / sqrt(d_k) to prevent the dot product from exploding
        b = torch.exp(a / K.shape[-1])
        # Sum over rows of the similarity matrices. Each row in c corresponds to
        # an attention head, and each column corresponds to the sum of a row in
        # the similarity matrices.
        c = torch.sum(b, dim=3)
        # Divide each row in the similarity matrices by their sum
        d = torch.einsum("bhij,bhi->bhij", b, 1 / c)
        # softmax(Q @ K.T) @ V for each attention head in each batch to get the scaled values
        S = torch.einsum("bhij,bhjk->bhik", d, V_proj)

        # Concatenate the scaled values for each attention head together and
        # project the resulting tensor into the original space
        B, h, n_v, d_v = S.shape
        concat = torch.permute(S, (0, 2, 1, 3)).reshape(B, n_v, h * d_v)
        return torch.einsum("bij,bjk->bik", concat, self.W_o)

In [102]:
m = MultiHeadAttention(1, 512, 64, 64, 8)
Q = torch.rand(1, 3, 512)
K = torch.rand(1, 5, 512)
V = torch.rand(1, 5, 512)
scaled = m.forward(Q, K, V)
assert scaled.shape == Q.shape